In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

TABELA_BRONZE = "workspace.bronze.inep_superior_cursos"
TABELA_SILVER = "workspace.silver.inep_superior_cursos"

VERSAO_TRANSFORMACAO = "inep_silver_v1"
ANOS_PROCESSAMENTO = [2021, 2022, 2023, 2024]

df_bronze = spark.table(TABELA_BRONZE)

In [0]:
%skip
# Campos essenciais para o problema de negócio
campos_essenciais = [
    "NU_ANO_CENSO",
    "TP_DIMENSAO",
    "TP_MODALIDADE_ENSINO",
    "SG_UF",
    "CO_UF",
    "CO_MUNICIPIO",
    "CO_IES",
    "CO_CURSO",
    "NO_CURSO",
    "CO_CINE_ROTULO",
    "NO_CINE_ROTULO",
    "CO_CINE_AREA_GERAL",
    "NO_CINE_AREA_GERAL",
    "CO_CINE_AREA_ESPECIFICA",
    "NO_CINE_AREA_ESPECIFICA",
    "CO_CINE_AREA_DETALHADA",
    "NO_CINE_AREA_DETALHADA",
    "QT_ING",
    "QT_MAT",
    "QT_CONC",
    "hash_origem",
    "dt_ingestao"
]

campos_ausentes = [
    campo for campo in campos_essenciais
    if campo not in df_bronze.columns
]

if campos_ausentes:
    raise ValueError(
        f"Campos essenciais ausentes: {campos_ausentes}"
    )

print("Todos os campos essenciais estão disponíveis.")

display(
    df_bronze
    .groupBy("NU_ANO_CENSO")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("hash_origem").alias("versoes_origem"),
        F.sum(
            F.when(F.col("CO_CINE_ROTULO").isNull(), 1)
            .otherwise(0)
        ).alias("cine_rotulo_nulos")
    )
    .orderBy("NU_ANO_CENSO")
)

In [0]:
%skip
df_diagnostico = (
    df_bronze
    .select(
        "NU_ANO_CENSO",
        "TP_DIMENSAO",
        "TP_MODALIDADE_ENSINO",
        "CO_CINE_ROTULO",
        "NO_CINE_ROTULO",
        "CO_CINE_AREA_GERAL",
        "NO_CINE_AREA_GERAL",
        "QT_ING",
        "QT_MAT",
        "QT_CONC"
    )
    .withColumn(
        "cine_rotulo_limpo",
        F.regexp_replace(
            F.trim(F.col("CO_CINE_ROTULO")),
            '^"+|"+$',
            ""
        )
    )
)

display(
    df_diagnostico
    .groupBy(
        "NU_ANO_CENSO",
        "TP_DIMENSAO",
        "TP_MODALIDADE_ENSINO"
    )
    .agg(
        F.count("*").alias("registros")
    )
    .orderBy(
        "NU_ANO_CENSO",
        "TP_DIMENSAO",
        "TP_MODALIDADE_ENSINO"
    )
)

display(
    df_diagnostico.select(
        "NU_ANO_CENSO",
        "CO_CINE_ROTULO",
        "cine_rotulo_limpo",
        "NO_CINE_ROTULO"
    ).limit(20)
)

In [0]:
from pyspark.sql import functions as F

# Recorte oficial do projeto
df_origem = df_bronze.filter(
    F.col("NU_ANO_CENSO").isin("2021", "2022", "2023", "2024")
)

# Nome original -> nome padronizado
mapeamento = {
    # Tempo e localização
    "NU_ANO_CENSO": "ano_censo",
    "NO_REGIAO": "nome_regiao",
    "CO_REGIAO": "codigo_regiao",
    "NO_UF": "nome_uf",
    "SG_UF": "sigla_uf",
    "CO_UF": "codigo_uf",
    "NO_MUNICIPIO": "nome_municipio",
    "CO_MUNICIPIO": "codigo_municipio",
    "IN_CAPITAL": "indicador_capital",
    "TP_DIMENSAO": "tipo_dimensao",

    # Curso e classificação CINE
    "CO_IES": "codigo_ies",
    "CO_CURSO": "codigo_curso",
    "NO_CURSO": "nome_curso",
    "CO_CINE_ROTULO": "codigo_cine_rotulo",
    "NO_CINE_ROTULO": "nome_cine_rotulo",
    "CO_CINE_AREA_GERAL": "codigo_cine_area_geral",
    "NO_CINE_AREA_GERAL": "nome_cine_area_geral",
    "CO_CINE_AREA_ESPECIFICA": "codigo_cine_area_especifica",
    "NO_CINE_AREA_ESPECIFICA": "nome_cine_area_especifica",
    "CO_CINE_AREA_DETALHADA": "codigo_cine_area_detalhada",
    "NO_CINE_AREA_DETALHADA": "nome_cine_area_detalhada",

    # Características educacionais
    "TP_MODALIDADE_ENSINO": "tipo_modalidade_ensino",
    "TP_GRAU_ACADEMICO": "tipo_grau_academico",
    "TP_NIVEL_ACADEMICO": "tipo_nivel_academico",
    "TP_REDE": "tipo_rede",
    "TP_CATEGORIA_ADMINISTRATIVA": "tipo_categoria_administrativa",
    "TP_ORGANIZACAO_ACADEMICA": "tipo_organizacao_academica",
    "IN_GRATUITO": "indicador_gratuito",

    # Indicadores
    "QT_CURSO": "quantidade_cursos",
    "QT_VG_TOTAL": "quantidade_vagas",
    "QT_INSCRITO_TOTAL": "quantidade_inscritos",
    "QT_ING": "quantidade_ingressantes",
    "QT_MAT": "quantidade_matriculas",
    "QT_CONC": "quantidade_concluintes",
    "QT_SIT_DESVINCULADO": "quantidade_desvinculados",
    "QT_SIT_TRANCADA": "quantidade_trancamentos",

    # Metadados
    "arquivo_origem": "arquivo_origem",
    "hash_origem": "hash_origem",
    "dt_ingestao": "dt_ingestao"
}

ausentes = [
    origem for origem in mapeamento
    if origem not in df_origem.columns
]

if ausentes:
    raise ValueError(f"Campos ausentes: {ausentes}")

df_silver = df_origem.select(
    *[
        F.col(origem).alias(destino)
        for origem, destino in mapeamento.items()
    ]
)

print(f"Colunas selecionadas: {len(df_silver.columns)}")

In [0]:
# 03 - Identificar os anos pendentes

# 1. Consultar as versões da Bronze
versoes_bronze = (
    df_bronze
    .filter(
        F.col("NU_ANO_CENSO").cast("int")
        .isin(ANOS_PROCESSAMENTO)
    )
    .groupBy("NU_ANO_CENSO")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("hash_origem").alias("qtd_versoes"),
        F.first("hash_origem").alias("hash_origem")
    )
    .collect()
)

controle_origem = {}

for linha in versoes_bronze:

    ano = int(linha["NU_ANO_CENSO"])

    if (
        linha["total_registros"] == 0
        or linha["qtd_versoes"] != 1
        or linha["hash_origem"] is None
    ):
        raise ValueError(
            f"Bronze inconsistente para o ano {ano}"
        )

    controle_origem[ano] = {
        "hash": linha["hash_origem"],
        "registros": linha["total_registros"]
    }

# Conferir se todos os anos esperados existem
anos_ausentes = (
    set(ANOS_PROCESSAMENTO) - set(controle_origem)
)

if anos_ausentes:
    raise ValueError(
        f"Anos ausentes na Bronze: {sorted(anos_ausentes)}"
    )

# 2. Consultar as versões da Silver
controle_silver = {}

if spark.catalog.tableExists(TABELA_SILVER):

    versoes_silver = (
        spark.table(TABELA_SILVER)
        .filter(
            F.col("ano_censo").isin(ANOS_PROCESSAMENTO)
        )
        .groupBy("ano_censo")
        .agg(
            F.count("*").alias("total_registros"),
            F.countDistinct("hash_origem").alias("qtd_hashes"),
            F.first("hash_origem").alias("hash_origem"),
            F.countDistinct("versao_transformacao").alias("qtd_regras"),
            F.first("versao_transformacao").alias("versao_regra")
        )
        .collect()
    )

    controle_silver = {
        int(linha["ano_censo"]): linha.asDict()
        for linha in versoes_silver
    }

# 3. Comparar Bronze e Silver
anos_pendentes = []

for ano in ANOS_PROCESSAMENTO:

    origem = controle_origem[ano]
    silver = controle_silver.get(ano)

    atualizado = (
        silver is not None
        and silver["total_registros"] == origem["registros"]
        and silver["qtd_hashes"] == 1
        and silver["hash_origem"] == origem["hash"]
        and silver["qtd_regras"] == 1
        and silver["versao_regra"] == VERSAO_TRANSFORMACAO
    )

    if atualizado:
        print(f"{ano}: Silver atualizada.")
    else:
        anos_pendentes.append(ano)
        print(f"{ano}: processamento necessário.")

print(f"\nAnos pendentes: {anos_pendentes}")

# 4. Encerrar quando não houver pendências
if not anos_pendentes:
    dbutils.notebook.exit("SEM_PENDENCIAS")

In [0]:
# 04 - Preparar DataFrame Silver

# Ler somente os anos que precisam ser processados
df_origem = df_bronze.filter(
    F.col("NU_ANO_CENSO").cast("int").isin(
        anos_pendentes
    )
)

# Garantir disponibilidade das colunas
ausentes = [
    origem
    for origem in mapeamento
    if origem not in df_origem.columns
]

if ausentes:
    raise ValueError(
        f"Campos ausentes na Bronze: {ausentes}"
    )

# Selecionar e renomear as colunas
df_silver = df_origem.select(
    *[
        F.col(origem).alias(destino)
        for origem, destino in mapeamento.items()
    ]
)

print(f"Anos em processamento: {anos_pendentes}")
print(f"Colunas selecionadas: {len(df_silver.columns)}")

In [0]:
from pyspark.sql import functions as F

# Tipos numéricos
campos_inteiros = [
    "ano_censo",
    "tipo_dimensao",
    "tipo_modalidade_ensino",
    "tipo_grau_academico",
    "tipo_nivel_academico",
    "tipo_rede",
    "tipo_categoria_administrativa",
    "tipo_organizacao_academica",
    "indicador_capital",
    "indicador_gratuito"
]

campos_quantidades = [
    "quantidade_cursos",
    "quantidade_vagas",
    "quantidade_inscritos",
    "quantidade_ingressantes",
    "quantidade_matriculas",
    "quantidade_concluintes",
    "quantidade_desvinculados",
    "quantidade_trancamentos"
]

# Converter valores numéricos, preservando nulos
for campo in campos_inteiros:
    df_silver = df_silver.withColumn(
        campo,
        F.expr(f"try_cast(trim(`{campo}`) AS INT)")
    )

for campo in campos_quantidades:
    df_silver = df_silver.withColumn(
        campo,
        F.expr(f"try_cast(trim(`{campo}`) AS BIGINT)")
    )

# Padronizar códigos CINE sem perder zeros à esquerda
campos_cine = [
    "codigo_cine_rotulo",
    "codigo_cine_area_geral",
    "codigo_cine_area_especifica",
    "codigo_cine_area_detalhada"
]

for campo in campos_cine:
    df_silver = df_silver.withColumn(
        campo,
        F.regexp_replace(
            F.trim(F.col(campo)),
            '^"+|"+$',
            ""
        )
    )

# Padronizar textos, sem alterar maiúsculas ou acentuação
campos_texto = [
    "nome_regiao",
    "nome_uf",
    "sigla_uf",
    "nome_municipio",
    "nome_curso",
    "nome_cine_rotulo",
    "nome_cine_area_geral",
    "nome_cine_area_especifica",
    "nome_cine_area_detalhada"
]

for campo in campos_texto:
    df_silver = df_silver.withColumn(
        campo,
        F.trim(F.col(campo))
    )

# Data efetiva da transformação
df_silver = df_silver.withColumn(
    "dt_transformacao",
    F.current_timestamp()
).withColumn(
    "versao_transformacao",
    F.lit(VERSAO_TRANSFORMACAO)
)

In [0]:
campos_numericos = {
    origem: destino
    for origem, destino in mapeamento.items()
    if destino in campos_inteiros + campos_quantidades
}

In [0]:
%skip
campos_verificacao = {
    "QT_ING": "quantidade_ingressantes",
    "QT_MAT": "quantidade_matriculas",
    "QT_CONC": "quantidade_concluintes",
    "QT_VG_TOTAL": "quantidade_vagas",
    "QT_CURSO": "quantidade_cursos"
}

df_comparacao = (
    df_origem
    .select(
        "NU_ANO_CENSO",
        *campos_verificacao.keys()
    )
    .withColumn(
        "_linha_id",
        F.monotonically_increasing_id()
    )
)

# Verificação direta na origem, sem depender de um join
expressoes = []

for campo in campos_verificacao:
    expressoes.append(
        F.sum(
            F.when(
                F.col(campo).isNotNull()
                & (F.trim(F.col(campo)) != "")
                & F.expr(
                    f"try_cast(trim(`{campo}`) AS BIGINT)"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(f"{campo}_invalidos")
    )

display(
    df_origem
    .groupBy("NU_ANO_CENSO")
    .agg(*expressoes)
    .orderBy("NU_ANO_CENSO")
)

In [0]:
%skip
# Colunas numéricas de origem e destino
campos_numericos = {
    origem: destino
    for origem, destino in mapeamento.items()
    if destino in campos_inteiros + campos_quantidades
}

validacoes = []

for origem, destino in campos_numericos.items():

    tipo = (
        "BIGINT"
        if destino in campos_quantidades
        else "INT"
    )

    valor = F.trim(F.col(origem))

    conversao_invalida = (
        valor.isNotNull()
        & (valor != "")
        & F.expr(
            f"try_cast(trim(`{origem}`) AS {tipo})"
        ).isNull()
    )

    validacoes.append(
        F.sum(
            F.when(conversao_invalida, 1).otherwise(0)
        ).alias(destino)
    )

df_validacao_tipos = (
    df_origem
    .groupBy("NU_ANO_CENSO")
    .agg(*validacoes)
    .orderBy("NU_ANO_CENSO")
)

display(df_validacao_tipos)

# Impedir gravação com conversões inválidas
totais = df_validacao_tipos.collect()

for linha in totais:
    for campo in campos_numericos.values():
        if linha[campo] != 0:
            raise ValueError(
                f"Ano {linha['NU_ANO_CENSO']}: "
                f"{linha[campo]} conversões inválidas em {campo}"
            )

print("Validação das conversões concluída.")

In [0]:
# 07 - Gravação incremental na Silver

anos_atualizados = []

for ano in anos_pendentes:

    print(f"\nProcessando ano {ano}...")

    df_ano = df_silver.filter(
        F.col("ano_censo") == ano
    )

    # Validar a contagem antes da gravação
    total_origem = controle_origem[ano]["registros"]
    total_silver = df_ano.count()

    if total_silver != total_origem:
        raise ValueError(
            f"{ano}: divergência antes da gravação. "
            f"Bronze={total_origem}, Silver={total_silver}"
        )

    # Primeira criação da tabela
    if not spark.catalog.tableExists(TABELA_SILVER):

        (
            df_ano.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(TABELA_SILVER)
        )

    # Atualização seletiva por ano
    else:

        (
            df_ano.write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"ano_censo = {ano}"
            )
            .option("mergeSchema", "true")
            .saveAsTable(TABELA_SILVER)
        )

    anos_atualizados.append(ano)

    print(f"{ano}: {total_silver:,} registros gravados.")

print(f"\nAnos atualizados: {anos_atualizados}")

In [0]:
df_bronze_contagem = (
    spark.table(TABELA_BRONZE)
    .filter(F.col("NU_ANO_CENSO").isin(
        "2021", "2022", "2023", "2024"
    ))
    .groupBy(
        F.col("NU_ANO_CENSO").cast("int").alias("ano_censo")
    )
    .agg(F.count("*").alias("registros_bronze"))
)

df_silver_contagem = (
    spark.table(TABELA_SILVER)
    .groupBy("ano_censo")
    .agg(
        F.count("*").alias("registros_silver"),
        F.countDistinct("hash_origem").alias("versoes_origem")
    )
)

df_reconciliacao = (
    df_bronze_contagem
    .join(df_silver_contagem, "ano_censo", "full")
    .withColumn(
        "diferenca",
        F.col("registros_bronze") - F.col("registros_silver")
    )
    .orderBy("ano_censo")
)

display(df_reconciliacao)

falhas = df_reconciliacao.filter(
    F.col("registros_bronze").isNull()
    | F.col("registros_silver").isNull()
    | (F.col("diferenca") != 0)
    | (F.col("versoes_origem") != 1)
).count()

if falhas:
    raise ValueError("Falha na reconciliação Bronze × Silver.")

print("Reconciliação Bronze × Silver concluída com sucesso.")